In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as func
import boto3

In [2]:
MINIO_ACCESS_KEY= "minioadmin"
MINIO_SECRET_KEY= "minioadmin"
MINIO_ENDPOINT = "http://minio:9000"
BUCKET_NAME= "datalake"
LOCAL_DATA_PATH= "/raw_mount"

In [3]:
spark= SparkSession.builder.appName("processVariant") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/27 04:06:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
var_sum= spark.read.parquet("s3a://datalake/raw_parquet/variant_summary_parquet")

25/11/27 04:06:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [5]:
var_sum.columns

['AlleleID',
 'Type',
 'Name',
 'GeneID',
 'GeneSymbol',
 'HGNC_ID',
 'ClinicalSignificance',
 'ClinSigSimple',
 'LastEvaluated',
 'RS# (dbSNP)',
 'nsv/esv (dbVar)',
 'RCVaccession',
 'PhenotypeIDS',
 'PhenotypeList',
 'Origin',
 'OriginSimple',
 'Assembly',
 'ChromosomeAccession',
 'Chromosome',
 'Start',
 'Stop',
 'ReferenceAllele',
 'AlternateAllele',
 'Cytogenetic',
 'ReviewStatus',
 'NumberSubmitters',
 'Guidelines',
 'TestedInGTR',
 'OtherIDs',
 'SubmitterCategories',
 'VariationID',
 'PositionVCF',
 'ReferenceAlleleVCF',
 'AlternateAlleleVCF',
 'SomaticClinicalImpact',
 'SomaticClinicalImpactLastEvaluated',
 'ReviewStatusClinicalImpact',
 'Oncogenicity',
 'OncogenicityLastEvaluated',
 'ReviewStatusOncogenicity',
 'SCVsForAggregateGermlineClassification',
 'SCVsForAggregateSomaticClinicalImpact',
 'SCVsForAggregateOncogenicityClassification']

In [6]:
var_sum.withColumn("PhenotypeList2", func.split(var_sum.PhenotypeList, "\\|")) \
        .select(func.explode("PhenotypeList2").alias("PhenotypeList2")).filter(func.col("PhenotypeList2").contains(";")).distinct().show(20, False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|PhenotypeList2                                                                                                                                                                                                                                                                                                                       |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Acute infantile

In [12]:
var_sum.select("VariationID",
        "GeneSymbol",
        "Chromosome",
        "Start",
        "Stop",
        "ReferenceAllele",
        "AlternateAllele",
        "PhenotypeList",
        "ReviewStatus",).show(5, False)

+-----------+----------+----------+--------+--------+---------------+---------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------+
|VariationID|GeneSymbol|Chromosome|Start   |Stop    |ReferenceAllele|AlternateAllele|PhenotypeList                                                                                       |ReviewStatus                                        |
+-----------+----------+----------+--------+--------+---------------+---------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------+
|2          |AP5Z1     |7         |4820844 |4820847 |na             |na             |Hereditary spastic paraplegia 48|Macular dystrophy with or without extraocular features|not provided|criteria provided, multiple submitters, no conflicts|
|2          |AP5Z1     |7         |47812

In [5]:
filtered = (
    var_sum
    .filter(func.col("PhenotypeList").isNotNull())
    .filter(func.col("GeneSymbol") != "-")
    .filter(func.length("PhenotypeList") > 0)
)


step= (
    filtered
    .withColumn("pheno_level1", func.split("PhenotypeList", "\\|"))
    .withColumn("pheno_level1", func.explode("pheno_level1"))

    .withColumn(
        "pheno_level2",
        func.split(func.col("pheno_level1"), r";(?=[A-Z])")
    )
    .withColumn("Disease", func.explode("pheno_level2"))
    .withColumn("Disease", func.trim("Disease"))
    .filter(func.col("Disease") != "")
    .filter(~func.col("Disease").isin(
        "not provided",
        "not specified",
        "not applicable",
        "see cases",
        "See cases",
        "-"
    ))
)
    
var_sum_cleaned= step.select(
        "VariationID",
        "GeneSymbol",
        "Chromosome",
        "Start",
        "Stop",
        "ReferenceAllele",
        "AlternateAllele",
        "Disease",
        "ReviewStatus",
    )
var_sum_cleaned= var_sum_cleaned.dropDuplicates().cache()

In [6]:
var_sum_cleaned.limit(5).toPandas()

,VariationID,GeneSymbol,Chromosome,Start,Stop,ReferenceAllele,AlternateAllele,Disease,ReviewStatus
0,83,HPSE2,10,98482733,98482733,na,na,Urofacial syndrome type 1,"criteria provided, multiple submitters, no con..."
1,123,CBS,21,44479409,44479409,na,na,"HYPERHOMOCYSTEINEMIA, THROMBOTIC, CBS-RELATED","criteria provided, single submitter"
2,132,CBS,21,43065481,43065481,na,na,"HYPERHOMOCYSTEINEMIA, THROMBOTIC, CBS-RELATED","criteria provided, multiple submitters, no con..."
3,179,OAT,10,124405551,124405551,na,na,Ornithine aminotransferase deficiency,"criteria provided, single submitter"
4,536,EYS,6,65057728,65320715,na,na,Retinitis pigmentosa 25,no assertion criteria provided


In [8]:
var_sum_cleaned.limit(20).toPandas()

,VariationID,GeneSymbol,Chromosome,Start,Stop,ReferenceAllele,AlternateAllele,Disease,ReviewStatus
0,83,HPSE2,10,98482733,98482733,na,na,Urofacial syndrome type 1,"criteria provided, multiple submitters, no con..."
1,123,CBS,21,44479409,44479409,na,na,"HYPERHOMOCYSTEINEMIA, THROMBOTIC, CBS-RELATED","criteria provided, single submitter"
2,132,CBS,21,43065481,43065481,na,na,"HYPERHOMOCYSTEINEMIA, THROMBOTIC, CBS-RELATED","criteria provided, multiple submitters, no con..."
3,179,OAT,10,124405551,124405551,na,na,Ornithine aminotransferase deficiency,"criteria provided, single submitter"
4,536,EYS,6,65057728,65320715,na,na,Retinitis pigmentosa 25,no assertion criteria provided
5,550,FECH,18,57580222,57580222,na,na,"Protoporphyria, erythropoietic, 1","criteria provided, conflicting classifications"
6,613,PAH,12,102851703,102851703,na,na,Phenylketonuria,"criteria provided, multiple submitters, no con..."
7,617,PAH,12,103234250,103234250,na,na,PAH-related disorder,reviewed by expert panel
8,633,PAH,12,103238111,103238111,na,na,Phenylketonuria,reviewed by expert panel
9,786,GPD1L,3,32159096,32159096,na,na,GPD1L-related disorder,"criteria provided, conflicting classifications"


In [10]:
(var_sum_cleaned
 .filter(
     (func.col("Disease") == "type 1") |
     (func.col("Disease") == "type I") |
     (func.col("Disease").rlike(r"^type\s*\d+$"))
 )
 .show(truncate=False)
)

+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+
|VariationID|GeneSymbol|Chromosome|Start|Stop|ReferenceAllele|AlternateAllele|Disease|ReviewStatus|
+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+
+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+



In [ ]:
print(var_sum_cleaned.filter(func.col("Disease")=="-").count())
var_sum_cleaned.count()

In [11]:
var_sum_cleaned.write.mode("overwrite").parquet("s3a://datalake/silver_layer/variant_summary_parquet")

In [140]:
silver_layer= spark.read.parquet("s3a://datalake/silver_layer/variant_summary_parquet")

In [142]:
silver_layer.filter(func.col("Disease").isNull()).show()
silver_layer.filter(func.col("GeneSymbol").isNull()).show()
silver_layer.filter(func.col("GeneSymbol")=="-").show()
silver_layer.filter(func.col("Disease")=="-").show()


+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+
|VariationID|GeneSymbol|Chromosome|Start|Stop|ReferenceAllele|AlternateAllele|Disease|ReviewStatus|
+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+
+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+

+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+
|VariationID|GeneSymbol|Chromosome|Start|Stop|ReferenceAllele|AlternateAllele|Disease|ReviewStatus|
+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+
+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+

+-----------+----------+----------+-----+----+---------------+---------------+-------+------------+
|VariationID|GeneSymbol|Chromosome|Start|Stop|ReferenceAllele|AlternateAllele|Disease|ReviewStatus

In [12]:
spark.stop()